# 00 - Preparar el proyecto

Este notebook deja todo listo para que los demás funcionen. Hace cuatro cosas:

1. Le dice a Python dónde está nuestro código (la carpeta `src/bondis`), para poder escribir `from bondis import ...`.
2. Crea en Databricks el lugar donde se guardan las tablas y los archivos (`bondis_stm.datos.raw`, definido en `config.py`).
3. Crea una carpeta para cada tipo de archivo (viajes, clima, etc.).
4. Muestra cómo dividimos los datos en train, val y test.

Se puede correr todas las veces que quieras. Si algo ya existe, no lo vuelve a crear ni lo borra.

No hace falta abrirlo a mano: los otros notebooks lo corren al principio con `%run ./00_setup`.

## 1. Encontrar nuestro código

Python solo puede importar código de las carpetas que conoce. Esta celda busca la carpeta del repositorio subiendo carpeta por carpeta hasta encontrar `src/bondis`, y la agrega a la lista de carpetas que Python conoce (`sys.path`).

In [0]:
import importlib
import os
import sys
 
def _buscar_raiz_repo(inicio):
    """Sube carpeta por carpeta desde 'inicio' hasta encontrar src/bondis.
 
    Así funciona igual si el notebook se corre solo o si se llama con %run desde otro.
    """
    actual = os.path.abspath(inicio)
    while True:
        if os.path.isdir(os.path.join(actual, "src", "bondis")):
            return actual
        padre = os.path.dirname(actual)
        if padre == actual:
            return None
        actual = padre
 
 
_contexto = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
_notebook_path = "/Workspace" + _contexto.notebookPath().get()
 
REPO_ROOT = _buscar_raiz_repo(os.getcwd()) or _buscar_raiz_repo(os.path.dirname(_notebook_path))
if REPO_ROOT is None:
    raise FileNotFoundError(
        "No se encontró la carpeta src/bondis.\n"
        f"  Directorio de trabajo: {os.getcwd()}\n"
        f"  Notebook: {_notebook_path}\n"
        "Verificar que el notebook esté dentro del Git folder del repo."
    )
SRC_PATH = os.path.join(REPO_ROOT, "src")
 
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)
 
from bondis import config
 
importlib.reload(config)  # vuelve a leer config.py por si lo cambiamos, sin reiniciar el cluster
 
# Mostramos la ruta desde la carpeta del repo (la ruta completa tiene el usuario de Databricks)
print("Código encontrado en:", os.path.relpath(SRC_PATH, os.path.dirname(REPO_ROOT)))
 
# Si este notebook se corre solo, muestra todo el detalle (sección 4). Si lo llama otro
# notebook con %run, no lista los archivos ni las tablas, para que sus salidas queden cortas.
# Con %run, la ruta que da el contexto es la del notebook que llamó, no la de este.
MOSTRAR_DETALLE = _notebook_path.rstrip("/").endswith("00_setup")

Código encontrado en: Trabajo-Aprendizaje-Automatico-I/src


## 2. Crear el catálogo, el esquema y el volume

Databricks ordena todo en tres niveles, como carpetas una dentro de otra:

- **Catálogo** (`bondis_stm`): la carpeta grande del proyecto.
- **Esquema** (`datos`): una subcarpeta donde van las tablas.
- **Volume** (`raw`): una carpeta para guardar archivos sueltos, como los ZIP y CSV que bajamos.


In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {config.CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {config.CATALOG}.{config.SCHEMA}")
spark.sql(
    f"CREATE VOLUME IF NOT EXISTS {config.CATALOG}.{config.SCHEMA}.{config.VOLUME}"
)
 
print(f"Catálogo y esquema listos: {config.CATALOG}.{config.SCHEMA}")
print(f"Volume listo:  {config.RAW_PATH}")

Catálogo y esquema listos: bondis_stm.datos
Volume listo:  /Volumes/bondis_stm/datos/raw


## 3. Crear una carpeta para cada tipo de archivo

| Carpeta | Qué va adentro |
|---|---|
| `viajes_zip/` | Los ZIP de viajes de cada mes, bajados a mano del portal de la Intendencia |
| `viajes/` | Los CSV de viajes, ya sacados de los ZIP |
| `clima/` | Lluvia y temperatura de cada hora |

El calendario (feriados, clases y paros) no tiene carpeta, porque está escrito a mano en `src/bondis/calendario.py`.

In [0]:
for carpeta in [config.RAW_ZIP, config.RAW_VIAJES, config.RAW_CLIMA]:
    dbutils.fs.mkdirs(carpeta)
    print("Carpeta lista:", carpeta)

Carpeta lista: /Volumes/bondis_stm/datos/raw/viajes_zip
Carpeta lista: /Volumes/bondis_stm/datos/raw/viajes
Carpeta lista: /Volumes/bondis_stm/datos/raw/clima


## 4. Revisar que todo esté bien

Muestra la versión de Spark y cuántos archivos hay en cada carpeta. Sirve para ver de un vistazo si falta subir algo.

Si este notebook lo llama otro con `%run`, solo muestra cuántos archivos hay en cada carpeta. La lista archivo por archivo y la tabla con las tablas del esquema aparecen solo cuando se corre el 00 solo. Así las salidas de los demás notebooks quedan cortas.

In [0]:
print("Versión de Spark:", spark.version)
print()
 
for carpeta in [config.RAW_ZIP, config.RAW_VIAJES, config.RAW_CLIMA]:
    archivos = [f for f in dbutils.fs.ls(carpeta) if not f.isDir()]
    total_mb = sum(f.size for f in archivos) / 1e6
    print(f"{carpeta}: {len(archivos)} archivo(s), {total_mb:.1f} MB")
    # La lista archivo por archivo solo aparece al correr este notebook solo (ver sección 1)
    if MOSTRAR_DETALLE:
        for f in archivos:
            print(f"    - {f.name}  ({f.size / 1e6:.1f} MB)")

Versión de Spark: 4.2.0

/Volumes/bondis_stm/datos/raw/viajes_zip: 34 archivo(s), 10152.3 MB
    - -cbMp2fwT6iwiXbVNYeWhw.zip  (293.9 MB)
    - 8Y3tuD7oRxGMFyKUQgMGhg.zip  (247.5 MB)
    - JieoWdKWTtCybN0OFdQP2Q.zip  (304.6 MB)
    - OPTIDr4sSSSzUy5ApwA8BQ.zip  (322.5 MB)
    - PVIk6skATgG_BCznG3neMQ.zip  (316.4 MB)
    - WURN2Wu6QZqtd9biBITFyA.zip  (311.3 MB)
    - WjYgv9t8Rv2CLPHmWo-uQw.zip  (349.4 MB)
    - aYTOp-sAQeqa_ov1lWrFBQ.zip  (329.4 MB)
    - i0qU2jRXTaqHBMi9_Tfg7A.zip  (237.5 MB)
    - qVGaCxW0QcWgJc7LIVv_OA.zip  (324.8 MB)
    - ufsChbV3Tb-HLZkJYW4Fww.zip  (289.0 MB)
    - viajes_octubre_de_2026.zip  (337.4 MB)
    - viajes_stm_012025.zip  (233.0 MB)
    - viajes_stm_012026.zip  (223.6 MB)
    - viajes_stm_022025.zip  (243.9 MB)
    - viajes_stm_022026.zip  (236.8 MB)
    - viajes_stm_032025.zip  (306.9 MB)
    - viajes_stm_032026.zip  (321.2 MB)
    - viajes_stm_042025.zip  (302.3 MB)
    - viajes_stm_042026.zip  (303.7 MB)
    - viajes_stm_052025.zip  (323.5 MB)
    - v

In [0]:
# Las tablas del esquema solo aparecen al correr este notebook solo (ver sección 1)
if MOSTRAR_DETALLE:
    display(spark.sql(f"SHOW TABLES IN {config.CATALOG}.{config.SCHEMA}"))

database,tableName,isTemporary
datos,calendario,false
datos,calidad_silver,false
datos,clima,false
datos,comparacion_validacion,false
datos,demanda_agregada,false
datos,demanda_gold,false
datos,lineas,false
datos,resultados_modelos,false
datos,viajes_bronze,false
datos,viajes_silver,false


## 5. Cómo dividimos los datos

Usamos las fechas para dividir, no el azar, porque queremos predecir el futuro con el pasado. Un año entero para cada cosa:

| Conjunto | Fechas | Para qué se usa |
|---|---|---|
| `train` | todo 2024 | El modelo aprende con estos datos |
| `val` | todo 2025 | Comparamos y elegimos los parámetros de la regresión |
| `test` | 2026 (lo que haya) | Probamos el modelo elegido una sola vez, al final |

La fecha de fin **no** está incluida. Por eso siempre se filtra con `fecha >= inicio` y `fecha < fin`.

`train_final` es train + val juntos. Cuando ya elegimos el modelo, lo volvemos a entrenar con 2024 y 2025 antes de probarlo en 2026.

In [0]:
print(f"{'Conjunto':<12} {'Inicio':<12} {'Fin (excl.)':<12}")
for nombre, (inicio, fin) in config.PARTICIONES.items():
    print(f"{nombre:<12} {inicio:<12} {fin:<12}")
print(f"{'train_final':<12} {config.PARTICION_TRAIN_FINAL[0]:<12} {config.PARTICION_TRAIN_FINAL[1]:<12}")
 
print()
print(f"Horizonte de predicción: {config.HORIZONTE_DIAS} día(s)")
print(f"Semilla: {config.SEED}")

Conjunto     Inicio       Fin (excl.) 
train        2024-01-01   2025-01-01  
val          2025-01-01   2026-01-01  
test         2026-01-01   2027-01-01  
train_final  2024-01-01   2026-01-01  

Horizonte de predicción: 1 día(s)
Semilla: 42
